In [1]:
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

# Download latest version
path = kagglehub.dataset_download("gregorut/videogamesales")

print("Path to dataset files:", path)

/kaggle/input/videogamesales/vgsales.csv
Path to dataset files: /kaggle/input/videogamesales


# Predict video game sales (Regression)

**Understand the Problem**

Define the task: Is it regression (predicting continuous values) or classification (predicting discrete labels)?
Ensure data contains input features (predictors) and target variable.

In [2]:
# Import helpful libraries
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# Load the data, and separate the target
video_game_file_path = '/kaggle/input/videogamesales/vgsales.csv'
video_game_data = pd.read_csv(video_game_file_path)

#Feature Selection: Eliminate irrelevant features to improve model efficiency

# Define target variable
y = video_game_data['Global_Sales']

# Create X with set of features
features = ['Genre', 'Publisher', 'Year', 'Rank']
X = video_game_data[features]

#Train-Test Split: Split data into training and validation sets. 

X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=1)

# 2. Data Preprocessing

#Handle Missing Data
cols_with_missing = [col for col in X_train.columns if X_train[col].isnull().any()]
#drop columns in training and validation data
X_train = X_train.drop(cols_with_missing, axis=1)
X_valid = X_valid.drop(cols_with_missing, axis=1)

#Encode Categorical Variables: Use methods like OrdinalEncoder or OneHotEncoder

# Get list of categorical variables
s = (X_train.dtypes == 'object')
object_cols = list(s[s].index)

print("Categorical variables:")
print(object_cols)

# Apply one-hot encoder to each column with categorical data
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(X_train[object_cols]))
OH_cols_valid = pd.DataFrame(OH_encoder.transform(X_valid[object_cols]))

# One-hot encoding removed index; put it back
OH_cols_train.index = X_train.index
OH_cols_valid.index = X_valid.index

# Remove categorical columns (will replace with one-hot encoding)
num_X_train = X_train.drop(object_cols, axis=1)
num_X_valid = X_valid.drop(object_cols, axis=1)

# Add one-hot encoded columns to numerical features
OH_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
OH_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

# Ensure all columns have string type
OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_valid.columns = OH_X_valid.columns.astype(str)


Categorical variables:
['Genre']


/opt/conda/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [3]:
# 3. Model Building & Training (Try both models)
# Select the model: Decision tree/Random Forest for regression or classification.
# Specify hyper parameters relevant to model: e.g. max_depth: Controls tree depth, n_estimators etc. 
# Fit on training data
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

# Random forest model
vg_model = RandomForestRegressor(random_state=1)
vg_model.fit(OH_X_train, y_train)


# # Decision tree model
# vg_model = DecisionTreeRegressor(random_state=1)
# vg_model.fit(X, y)

RandomForestRegressor(random_state=1)

In [4]:
# 4. Model Evaluation - Evaluate on the training and validation data

vg_val_predictions = vg_model.predict(OH_X_valid)
vg_val_mae = mean_absolute_error(vg_val_predictions, y_valid)
print(vg_val_mae)


# For Classification: Metrics: Accuracy, Confusion matrix to understand prediction distribution.
# For Regression: Metrics: Mean Absolute Error (MAE)
# Comment on whether your model is underfitting or overfitting or not.

0.002555903614458157


In [5]:
# Visualize the first tree in the forest
plt.figure(figsize=(20, 10))
#plot_tree(.estimators_[25], feature_names=X_train.columns, impurity = False, class_names=['No', 'Yes'], filled=True)
#plt.show()

<Figure size 2000x1000 with 0 Axes>

<Figure size 2000x1000 with 0 Axes>